<h1>------------------------------------------------</h1>
<h1>IV. Evaluation of ensemble solar forecast quality</h1>
<h1>------------------------------------------------</h1>

Like deterministic forecast, the quality of an ensemble forecast is the correspondence with corresponding observation. However, a probabilistic forecast contains more information than a detreministic one and the evaluation tools are more complex to handle. Firts, the quality of a probabilistic forecast relies on 2 fundamentals properties:
* Reliability (or calibration): Statistical consistency between
the forecasts and the observations (i.e. the nominal coverage
rate of the prediction intervals should be equal to the observed one)
* Resolution: It measures the capacity of a forecasting model to issue forecasts that are case-dependent.

Sharpness is also a very important feature of a probabilitic forecast. Users prefer to have sharp forecast (narrow prediction intervals). However, sharpness is not a quality indicator as it depends on the forecast itself (nà link with observations).

The verification tools proposed in this Jupyter Notebook where developed by the [laboratory PIMENT](https://piment.univ-reunion.fr/), University of La Reunion. They are detailed in the following article:

-------
<i>Lauret, P., David, M., & Pinson, P. (2019). Verification of solar irradiance probabilistic forecasts. Solar Energy, 194, 254–271. https://doi.org/10.1016/j.solener.2019.10.041</i>


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import os

<h1>------------------------------------------------------</h1>
<h1>Section A: Evaluation of the raw ECMWF-EPS</h1>

The raw ECMWF-EPS is the GHI ensemble forecast provided by ECMWF without any tranformations.
<h2>Load and read the data</h2>

The data used in this exercice are publicly available in a GitHub repository of the research lab. PIMENT, University of La Reunion ([Probabilistic_solar_forecast_lesson](https://github.com/Laboratoire-Piment/Probabilistic_solar_forecast_lesson)). If you copy and use this Jupyter Notebook on your computer, adapt the path to the data file.

We will use a NetCDF file containing the 51 EPS members of ECMWF, the corresponding observations and the clear sky conditions, covering 6 month from July 1st to December 31st 2022. The forecast cover a single location, the Terre sainte campus in La Reunion, where high quality solar measurments are available (21.333°S, 51,483°E, 75m, UTC+4h).

In [ ]:
# Dowload the required data from GitHub (https://github.com/Laboratoire-Piment/Probabilistic_solar_forecast_lesson)
destination = "temp/nwp_eps_1h.nc"
fs = fsspec.filesystem("github", org="Laboratoire-Piment", repo="Probabilistic_solar_forecast_lesson")
fs.get("Data/nwp_eps_1h.nc", destination)

# Read and display the file content
ds_eps = xr.open_dataset("temp/nwp_eps_1h.nc")
ds_eps

<h2>Information on NetCDF file</h2>

<h3>Dimensions:</h3>

* <b>basetime</b>: Representing the time when the forecast is done
* <b>step</b>: Steps indicating the time steps from the base time, also called horizon
* <b>member</b>: Number of the EPS members

<h3>Coordinates:</h3>

* <b>basetime</b>: A datetime coordinate
* <b>step</b>: A timedelta coordinate representing the time steps after the base time
* <b>member</b>: A integer representing the number of the EPS members
* <b>latitude</b>: Gives the value in degrees of the location latitude
* <b>longitude</b>: Gives the value in degrees of the location longitude

<h3>Data Variables:</h3>

* <b>GHI_cf</b>: Forecasted GHI ($W.m^{-2}$) of the control forecast for each of combination of `basetime` and `step`.
* <b>GHI_pf</b>: Forecasted GHI ($W.m^{-2}$) of the perturbed forecasts for each combination of `basetime`, `step`  and `memeber`.
* <b>GHI_clear</b>: GHI ($W.m^{-2}$) from Copernicus Atmosphere Monitoring Service (CAMS) McClear Clear-Sky for each combination of `basetime` and `step`.
* <b>GHI_meas</b>: Measured GHI ($W.m^{-2}$) for each combination of `basetime` and `step`.
* <b>zenith</b>: Solar zenith angle (°) for each combination of `basetime` and `step`.

<h2>Filtering of nigth GHI values</h2>

A specificity of solar irradiance is the intermittency between days and nights. Forecasting night values (no sun) is very simple and results in perfect precictions. It is thus important to not take into account for night values in the evaluation process. The zenith angle, which give the angle between the sun and the zenith, ranges between 0° and 90° during daytime. To remove zero night values of GHI, we filter the dataset with the zenith angle.

Question
* What is a good value of the zenith angle to filter the data?

Below, we will only use the filtered dataset.

In [ ]:
# Define solar zenith angle threshold
max_zenith = 

# Create a filtered dataset ds_det_f
ds_eps = ds_eps.where(ds_eps['zenith'] <= max_zenith).dropna(dim='step')

<h2>Exercice A.1: Evaluation of the reliability with the rank histogram</h2>

A rank histogram is a visual tool sepcifically designed to assess the reliability (or calibration) of ensemble forecasts. Rank histograms permit to assess the statistical consistency of the ensemble, that is, if the observation can be seen statistically just like another member of the ensemble. The flatness of the rank histogram indicates that the ensemble members are statistically indistinguishable from the observations (good calibration). An under-dispersed ensemble (i.e. ensemble dispersion consistently too small) leads to a U-shape rank histogram and shows that the observation will often be an outlier in the distribution of ensemble members. Conversely, an over-dispersed ensemble (i.e. ensemble dispersion consistently too large) gives a hump shape rank histogram and indicates that the observation may too often be in the middle of the ensemble distribution.

<h3>Function to plot a rank histogram<h3>

In [ ]:
def rank_hist(eps,obs,plot_title):
  # INPUTS
  #   eps: DataFrame with the ensemble members of size (steps,members)
  #   obs: Dataframe of observations of size (steps,1)
  #   plot_title: Title of the figure (string)
  # OUTPUT
  #   rank_freq: Frequency ratio of the ranks

  n,m = np.shape(eps) # n: number of steps, m: number of members
  rank_count = np.zeros([m+1]) # Vector to store the number of observation occurences per rank

  for i in np.arange(n):
    s = np.sort(np.concat((eps.iloc[i,:], obs.iloc[i]))) # Sort EPS members and observation
    idx_obs = np.argmin(np.abs(s-obs.iloc[i].values)) # Find position of the obs
    rank_count[idx_obs] = rank_count[idx_obs]+1 # Count rank occurences of observation

  rank_freq = rank_count/n # Convert count to relative frequency

  perfect_freq = np.ones([m+1])/(m+1) # Line of perfect calibration

  # Plot rank histogram
  plt.rcParams.update({'font.size': 10})
  plt.figure(figsize=[10, 3])
  plt.bar(np.arange(m+1), rank_freq, color='white', edgecolor='black', width=1.0) # Plot rank histogram
  plt.plot(np.arange(m+1), perfect_freq, linestyle='--', linewidth=1, color='#555555') # Add the perfect frequency line
  plt.title(plot_title)
  plt.xlabel('Rank')
  plt.ylabel('Relative frequency')
  plt.yscale('log') # Use a logarithmic scale for the y axis

  return rank_freq

<h3>Plot the rank histogram of the raw ECMWF-EPS</h3>

Use the function above to plot the rank histogram for all the forecast/observation pairs:
* Gather all the forecast members (control forecast + perturbed forecasts) in a single DataFrame with
* Gather corresponding GHI observation in a single DataFrame
* Plot the rank histogram using the function rank_hist

Questions
* What is the global shape of the rank histogram?
* Are the ranks close to the perfect frequency line?
* Is the raw ECMWF-EPS reliable?



In [ ]:
# Garther the forecast members in the DataFrame df_eps
df_eps = pd.DataFrame(np.reshape(ds_eps['GHI_pf'].values,
                                 (ds_eps.sizes['basetime']*ds_eps.sizes['step'],
                                  ds_eps.sizes['member'])))

# Garther the observations in the DataFrame df_obs
df_obs = pd.DataFrame(np.reshape(ds_eps['GHI_meas'].values,
                                 (ds_eps.sizes['basetime']*ds_eps.sizes['step'],1)))

# Plot the rank histogram
rk = rank_hist(df_eps,df_obs,'Raw ECMWF-EPS')

<h2>Exercice A.2: Evaluate the quality of the EPS with the Continuous Rank Probability Score (CRPS)</h2>

Numerous metrics exist to evaluate the quality of probabilistic forecast (e.g. CRPS, pinball loss, ignorance score, interval score, etc.). However, the CRPS is the most used because it has several very intersting chacteristics:
* For deterministic forecast, the CRPS is equal to the MAE (Mean Absolute Error),
* The CRPS is a striclty proper score,
* The CRPS can be decomposed in reliability and the resolution.

The general formulation of the CRPS is:

$$CRPS = \frac{1}{N} \sum^N_{i=1} \int_{-\infty}^{+\infty} [\hat{F}_{fcst}^i - F_{obs}^i]^2 dx$$

where N is the number of forecast/observation pairs,  $\hat{F}_{fcst}$ the CDF of the forecast and $F_{obs}$ the CDF of the observation.

The decomposition of the CRPS gives:

$$CRPS = RELIABILITY - RESOLUTION + UNCERTAINTY$$

The uncertainty term cannot be modified by the forecast system and depends only on the observations variability.

As an ensemble is a set of discrete quantile/probability pairs, it is not straightforward to apply the general formulation of the CRPS. Hersbach (see article below) proposed a method to compute and decomposed the CRPS using the classical definition of the ensemble CDF (see part III). Hersbach's method for calculating CRPS is by far the most widely used. You can find it the python library [CRPS](https://pypi.org/project/CRPS/) or in the R package [verification](https://cran.r-project.org/web/packages/verification/index.html). However, to my best knowledge, none of them provides the right CRPS decomposition for an ensemble forecast.

In the library of functions `crps_ur.py`, the function `crps_ensemble` provides the computation and the decomposition of the CRPS for ensemble forecasts accordingly to Hersbach's work.

--------
<i>Hersbach, H. (2000). Decomposition of the Continuous Ranked Probability Score for Ensemble Prediction Systems. Weather and Forecasting, 15(5), 559–570. https://doi.org/10.1175/1520-0434(2000)015%253C0559:DOTCRP%253E2.0.CO;2
</i>

In [ ]:
# Import the library crps_ur from GitHub (https://github.com/Laboratoire-Piment/Probabilistic_solar_forecast_lesson)
destination = "crps_ur.py"
fs = fsspec.filesystem("github", org="Laboratoire-Piment", repo="Probabilistic_solar_forecast_lesson")
fs.get("Jupyter_Notebooks/crps_ur.py", destination)

from crps_ur import crps_ensemble

# print(os.listdir())

<h2>Computation of the CRPS for a reduced set of forecast/observation pairs</h2>

To understand the behavior of the CRPS, we will test it on a fictive and simple set of forecast/observation pairs with 4 outcomes. The aim is to understand the meaning of the CRPS values and of its decomposition in Reliability, Resolution and Uncertainty.

Questions
* Compute the CRPS and its decomposition for the proposed set of forecast/observation pairs. What can you say about the ensemble forecast and the observations? What is the consequence for the Resolution and Uncertainty?
* Slightly change the values of the observation while keeping the ensemble forecast constant. What elements of the CRPS have changed? What can be said about the resolution?
* Now, change the values of the forecast members. Does the Uncertainty change? Which element is most influenced?

In [ ]:
# Create a fictive ensemble forecast with 3 members and 4
eps = pd.DataFrame([[1,2,4],[1,2,4],[1,2,4],[1,2,4]])

# Create the corresponding 4 observations
obs = pd.DataFrame([2, 2, 2, 2])

df_crps, df_crps_val = crps_ensemble(eps,obs)

print(df_crps)
print('---------------------------')
print('Rel - Res + Unc = ', df_crps['Reliability'] - df_crps['Resolution'] + df_crps['Uncertainty'])
print('---------------------------')
print('CRPS values:\n', df_crps_val)

<h3>Exrecice A.3: Compute the CRPS of the raw ECMWF-EPS</h3>

To asses the quality of raw ECMWF-EPS:
* Gather the ensemble forecasts and the corresponding observations in 2 DataFrames (tables)
* Compute the overall CRPS
* Plot the evolution of the CRPS and the relative CRPS over the step dimension

Questions
* What are the units of the CRPS and its components?
* Does the raw ECMWF-EPS have a good resolution?
* Is the raw ECMWF-EPS reliable?
* Why the plot of the evolution of CRPS and the relative CRPS are so different?


In [ ]:
# Garther all perturbed forecasts in the DataFrame df_pf
df_pf = pd.DataFrame(np.reshape(ds_eps['GHI_pf'].values,
                                 (ds_eps.sizes['basetime']*ds_eps.sizes['step'],
                                  ds_eps.sizes['member'])))

# Garther all control forecasts in the DataFrame df_cf
df_cf = pd.DataFrame(np.reshape(ds_eps['GHI_cf'].values,
                                 (ds_eps.sizes['basetime']*ds_eps.sizes['step'])))

# Combine df_pf and df_cf
df_eps = pd.concat([df_pf,df_cf], axis=1)

# Garther the observations in the DataFrame df_obs
df_obs = pd.DataFrame(np.reshape(ds_eps['GHI_meas'].values,
                                 (ds_eps.sizes['basetime']*ds_eps.sizes['step'],1)))

df_crps, df_crps_val = crps_ensemble(df_eps,df_obs)

print(df_crps)
print('---------------------------')
print('Rel - Res + Unc = ', df_crps['Reliability'] - df_crps['Resolution'] + df_crps['Uncertainty'])
print('---------------------------')
print('CRPS values:\n', df_crps_val)

# Plot the evolution of the relative CRPS over the step dimension
df_crps_step = pd.DataFrame(columns = ['step','CRPS','Relative CRPS'])

for i,st in enumerate(ds_eps.step):
  df_eps = pd.DataFrame(ds_eps['GHI_cf'].sel(step=st).values)
  df_obs = pd.DataFrame(ds_eps['GHI_meas'].sel(step=st).values)
  avg_ghi = np.nanmean(df_obs)
  df_crps_step.loc[i,'step'] = st/(10**9*3600)
  dum_crps, crps_val = crps_ensemble(df_eps,df_obs)  # Computre CRPS
  df_crps_step.loc[i,'CRPS'] = dum_crps['Mean CRPS'].values
  df_crps_step.loc[i,'Relative CRPS'] = df_crps_step.loc[i,'CRPS']/avg_ghi*100 # Transform in relative CRPS

# Plotting
plt.rcParams.update({'font.size': 10}) # Set font size
fig, (ax1, ax2) = plt.subplots(2, sharex=True, figsize=(10, 4))
ax1.plot(df_crps_step['step'], df_crps_step['CRPS'],'o')
ax1.set_ylabel('CRPS ($W.m^{-2}$)')
ax1.grid()

ax2.plot(df_crps_step['step'], df_crps_step['Relative CRPS'],'o')
ax2.set_ylabel('CRPS (%)')
ax2.set_xlabel('Lead time (hour)')
ax2.grid()

<h1>------------------------------------------------------</h1>
<h1>Section B: ECMWF-EPS calibration</h1>

The rank histogram of Exercise 1A showed that the raw ECMWF-EPS is unreliable In other word, the ECMWF-EPS is not well calibrated. The U-shape of the rank histogram indicates that the ensemble members are underdispersed (i.e. lack of spread), with observations often falling outside the ensemble interval. This a very common result for NWP ensemble forecast.

Numerous methods of post-processing were developed to calibrate ensemble forecasts. In this section, we will test one of them, the <b>Variance Deficit (VD)</b> method (see article below). The underlying assumption of the VD is simple: an ensemble suffers from a lack of spread because the variance of the its members is too low. The calibration process will increase the variance of the ensemble to have a good spread of the members.

In the library of functions `variance_deficit.py`, we implemented the functions required to run the VD method:
* `variance_deficit.train(fcst,obs)`: compute the VD coefficient on a training set of forecast/observation pairs
* `variance_deficit.predict(fcst,vd_coeff)`: calibrate a test set of forecasts

________
<i>Sperati, S., Alessandrini, S., & Delle Monache, L. (2016). An application of the ECMWF Ensemble Prediction System for short-term solar power forecasting. Solar Energy, 133, 437–450. https://doi.org/10.1016/j.solener.2016.04.016
</i>

In [ ]:
# Import the library variance_deficit from GitHub (https://github.com/Laboratoire-Piment/Probabilistic_solar_forecast_lesson)
destination = "variance_deficit.py"
fs = fsspec.filesystem("github", org="Laboratoire-Piment", repo="Probabilistic_solar_forecast_lesson")
fs.get("Jupyter_Notebooks/variance_deficit.py", destination)

import variance_deficit

<h2>Exercice B.1: Post-processing of ECMWF-EPS with the Variance Deficit (VD) method</h2>

To calibrate the raw ECMWF-EPS:
* Gather the ensemble forecasts and the corresponding observations in 2 DataFrames (tables df_eps and df_obs)
* Divide the 2 tables in training and test sets (e.g. 60% training and 40% test)
* Apply the VD method
* Compare visually 1 basetime run of the raw and calibarted ensemble

Questions
* What does mean the value of the VD coefficient?
* Does the calibrated ensemble trajectories (members) seems consistent with possible weather evolutions?

In [ ]:
# Garther all perturbed forecasts in the DataFrame df_pf
df_pf = pd.DataFrame(np.reshape(ds_eps['GHI_pf'].values,
                                 (ds_eps.sizes['basetime']*ds_eps.sizes['step'],
                                  ds_eps.sizes['member'])))

# Garther all control forecasts in the DataFrame df_cf
df_cf = pd.DataFrame(np.reshape(ds_eps['GHI_cf'].values,
                                 (ds_eps.sizes['basetime']*ds_eps.sizes['step'])))

# Combine df_pf and df_cf
df_eps = pd.concat([df_pf,df_cf], axis=1)

# Garther the observations in the DataFrame df_obs
df_obs = pd.DataFrame(np.reshape(ds_eps['GHI_meas'].values,
                                 (ds_eps.sizes['basetime']*ds_eps.sizes['step'],1)))

# Divide in training and test set
pct = 0.6 # Percentage of data of the train set
sz_train = pct*df_eps.shape[0]
df_eps_train = df_eps.loc[:sz_train-1, :]
df_obs_train = df_obs.loc[:sz_train-1, :]
df_eps_test = df_eps.loc[sz_train:, :]
df_obs_test = df_obs.loc[sz_train:, :]

print('-----------------')
print('Size train set:', df_eps_train.shape[0], ' / Size test set:', df_eps_test.shape[0])

# Apply the VD method
vd_coeff = variance_deficit.train(df_eps_train, df_obs_train)
print('-----------------')
print('vd_coeff:', vd_coeff)
print('-----------------')

# Apply the VD to the whole dataset
df_cal_eps = variance_deficit.predict(df_eps, np.sqrt(vd_coeff))

# Results an the test set alone
df_cal_eps_test =  df_cal_eps.loc[sz_train:, :]

In [ ]:
# Plotting one basetime before and after calibartion

# Select starting row of the df_eps_test and df_cal_eps
bs = 60

plt.rcParams.update({'font.size': 10})
plt.figure(figsize=[10, 3])
for m in np.arange(50):
    df_eps.iloc[bs:(bs+48),m].plot(color='grey', linewidth=0.5)
    df_cal_eps.iloc[bs:(bs+48),m].plot(color='blue', linewidth=0.25)
df_obs.iloc[bs:(bs+48),0].plot(color='magenta', linewidth=2, linestyle='dashed')
plt.ylabel('GHI ($W.m^{-2}$)')
plt.grid()

<h2>Visual assement with prediction intervals</h2>

Plotting all the trajectories on the same figure leads to something relatively unclear (sketchy). To better vizualize the change brought by the VD calibartion, we can plot the prediction intervals before and after calibration:
* Select a basetime and a window size (number of hours to plot)
* Plot the prédiction intervals of the ECMWF-EPS before and aftert the VD calibration

Questions
* What is the main transformation done by the VD of the prediction intervals?
* Does calibration seem to result in a better quality in our case?

In [ ]:
# Select basetime
bs = 
ws =  # Window size

# Observations
obs = df_obs.iloc[bs:(bs+ws),0]

# x axis
x = np.arange(ws)

# Cumulative probabilities
tau = np.arange(ds_eps['member'].shape[0]+1)/(ds_eps['member'].shape[0])

# Before calibration
eps_before = df_eps.iloc[bs:(bs+ws),:]
eps_before = np.sort(eps_before, axis=1) # Rank

# After calibration
eps_after = df_cal_eps.iloc[bs:(bs+ws),:]
eps_after = np.sort(eps_after, axis=1) # Rank

# Plotting
plt.rcParams.update({'font.size': 10}) # Set font size
fig, (ax1, ax2) = plt.subplots(2, sharex=True, figsize=(10, 4))
for quant in np.arange(0,25,5):
    lab = str(tau[quant]) + '-' + str(tau[50-quant])
    ax1.fill_between(x, eps_before[:,quant], eps_before[:,50-quant], color=str((5 - quant/5) / 6), label=lab)
ax1.plot(x, obs, color='cyan', linewidth=2, linestyle='dashed', label='Observation')
ax1.set_ylabel('GHI ($W.m^{-2}$)')
ax1.set_title('raw ECMWF-EPS')
ax1.grid()
ax1.legend()

for quant in np.arange(0,25,5):
    lab = str(tau[quant]) + '-' + str(tau[50-quant])
    ax2.fill_between(x, eps_after[:,quant], eps_after[:,50-quant], color=str((5 - quant/5) / 6), label=lab)
ax2.plot(x, obs, color='cyan', linewidth=2, linestyle='dashed', label='Observation')
ax2.set_ylabel('GHI ($W.m^{-2}$)')
ax2.set_xlabel('Step (hour)')
ax2.set_title('ECMWF-EPS after VD calibration')
ax2.grid()

<h2>Exercice B.2: Plot the rank histogram of the calibrated ECMWF-EPS and compute the CRPS</h2>

To further understand the effect of the VD calibration, we will now use the visual and quantitative evaluation tools presented previously:
* Plot the rank histogram for the training set, the test set and the whole dataset
* Compute the CRPS and its decomposition for the training set, the test set and the whole dataset

Questions
* Compare the rank histogram of the raw and calibrated ECMWF-EPS. Does VD method improve the calibation?
* Compare the CRPS (and its components) of the raw and calibrated ECMWF-EPS. Does the VD method improved the overall quality of the forecast?

In [ ]:
# Plot rank histogram of the training set
rk_train = rank_hist(df_cal_eps_train,df_obs_train,'Calibrated ECMWF-EPS - Training set')

# Plot rank histogram of ttest set
rk_test = rank_hist(df_cal_eps_test,df_obs_test,'Calibrated ECMWF-EPS - Test set')

# Plot rank histogram of the whole dataset
rk = rank_hist(df_cal_eps,df_obs,'Calibrated ECMWF-EPS - Whole dataset')

In [ ]:
# Compute the CRPS for training set
print('Training set')
df_crps, df_crps_val = crps_ensemble(df_eps_train,df_obs_train)
print('Before calibration')
print(df_crps)
df_crps, df_crps_val = crps_ensemble(df_cal_eps_train,df_obs_train)
print('After calibration')
print(df_crps)
print('---------------------------')

# Compute the CRPS for test set
print('Test set')
df_crps, df_crps_val = crps_ensemble(df_eps_test,df_obs_test)
print('Before calibration')
print(df_crps)
df_crps, df_crps_val = crps_ensemble(df_cal_eps_test,df_obs_test)
print('After calibration')
print(df_crps)
print('---------------------------')

# Compute the overall CRPS
print('Whole dataset')
df_crps, df_crps_val = crps_ensemble(df_eps,df_obs)
print('Before calibration')
print(df_crps)
df_crps, df_crps_val = crps_ensemble(df_cal_eps,df_obs)
print('After calibration')
print(df_crps)
print('---------------------------')